# Notebook 06 — Model 2: LightGBM

## Objective

Build a strong LightGBM multiclass classifier for SME Financial Health prediction, using leakage-safe categorical encoding and the engineered features developed in Notebook 05.

This notebook has two goals:

1. Produce a competitive standalone LightGBM model
2. Create a complementary model to CatBoost for later ensembling

Unlike CatBoost, LightGBM does not natively handle raw string categoricals in the same way, so categorical encoding must be handled carefully inside each cross-validation fold to avoid leakage.

In [2]:
# Imports and setup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.base import clone

import lightgbm as lgb

from src.preprocessing import PreprocessConfig, preprocess_train_test

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

In [3]:
# Load raw data and apply preprocessing pipeline

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

cfg = PreprocessConfig(id_col="ID", target_col="Target")

# LightGBM-friendly preprocessing: categorical missing -> "missing"
train, test = preprocess_train_test(train_raw, test_raw, cfg, for_model="lightgbm")

TARGET = cfg.target_col
ID = cfg.id_col

y = train[TARGET].copy()
X = train.drop(columns=[TARGET]).copy()

print("train shape:", train.shape)
print("test shape :", test.shape)

train shape: (9618, 47)
test shape : (2405, 46)


In [4]:
# Redefine the engineered features from Notebook 05

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def yn_to01(s):
        s = s.astype("string").str.lower().fillna("missing")
        return s.isin(["yes", "y", "1", "true"]).astype(int)

    # Access score
    access_cols = [c for c in [
        "has_credit_card",
        "has_debit_card",
        "has_loan_account",
        "has_internet_banking",
        "has_mobile_money",
    ] if c in df.columns]

    for c in access_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if access_cols:
        df["access_score"] = df[[f"{c}__01" for c in access_cols]].sum(axis=1)

    # Insurance score
    insurance_cols = [c for c in [
        "funeral_insurance",
        "medical_insurance",
        "motor_vehicle_insurance",
        "has_insurance",
    ] if c in df.columns]

    for c in insurance_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if insurance_cols:
        df["insurance_score"] = df[[f"{c}__01" for c in insurance_cols]].sum(axis=1)

    # Stress score
    stress_cols = [c for c in [
        "current_problem_cash_flow",
        "attitude_worried_shutdown",
        "problem_sourcing_money",
    ] if c in df.columns]

    for c in stress_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if stress_cols:
        df["stress_score"] = df[[f"{c}__01" for c in stress_cols]].sum(axis=1)

    # Formalization score
    formal_cols = [c for c in [
        "keeps_financial_records",
        "compliance_income_tax",
    ] if c in df.columns]

    for c in formal_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if formal_cols:
        df["formalization_score"] = df[[f"{c}__01" for c in formal_cols]].sum(axis=1)

    # Numeric ratios
    if "business_turnover" in df.columns and "business_expenses" in df.columns:
        turn = pd.to_numeric(df["business_turnover"], errors="coerce")
        exp  = pd.to_numeric(df["business_expenses"], errors="coerce")

        df["turnover_minus_expenses"] = turn - exp
        df["turnover_to_expenses"] = turn / exp.replace(0, np.nan)
        df["turnover_to_expenses"] = df["turnover_to_expenses"].replace([np.inf, -np.inf], np.nan)

    # Age bucket
    if "business_age_total_months" in df.columns:
        years = pd.to_numeric(df["business_age_total_months"], errors="coerce") / 12.0
        age_bucket = pd.cut(
            years,
            bins=[-np.inf, 0.5, 2, 5, 10, np.inf],
            labels=["<6m", "0.5-2y", "2-5y", "5-10y", "10y+"]
        )
        df["age_bucket"] = age_bucket.astype("string").fillna("missing")

    return df

X_fe = add_features(X).replace({pd.NA: np.nan})
test_fe = add_features(test).replace({pd.NA: np.nan})

print("X_fe shape:", X_fe.shape)
print("test_fe shape:", test_fe.shape)

X_fe shape: (9618, 67)
test_fe shape: (2405, 67)


In [5]:
# Identify categorical and numeric columns

from pandas.api.types import is_numeric_dtype

cat_cols = [c for c in X_fe.columns if not is_numeric_dtype(X_fe[c])]
num_cols = [c for c in X_fe.columns if is_numeric_dtype(X_fe[c])]

print("Categorical columns:", len(cat_cols))
print("Numeric columns:", len(num_cols))
print(cat_cols[:15])

Categorical columns: 33
Numeric columns: 34
['ID', 'country', 'attitude_stable_business_environment', 'attitude_worried_shutdown', 'compliance_income_tax', 'perception_insurance_doesnt_cover_losses', 'perception_cannot_afford_insurance', 'motor_vehicle_insurance', 'has_mobile_money', 'current_problem_cash_flow', 'has_cellphone', 'owner_sex', 'offers_credit_to_customers', 'attitude_satisfied_with_achievement', 'has_credit_card']


In [6]:
# Encode the target labels
# LightGBM expects numeric labels for multiclass classification.

from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
y_enc = pd.Series(label_enc.fit_transform(y), index=y.index)

class_names = list(label_enc.classes_)
class_names

['High', 'Low', 'Medium']

**Helper encoders**

We’ll implement:

- frequency encoding

- out-of-fold target encoding

- one-hot encoding for low-cardinality categoricals

In [7]:
# Frequency encoding

def add_frequency_encoding(train_df, valid_df, test_df, columns):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for c in columns:
        freq = train_df[c].fillna("missing").value_counts(normalize=True)
        train_df[f"{c}__freq"] = train_df[c].fillna("missing").map(freq)
        valid_df[f"{c}__freq"] = valid_df[c].fillna("missing").map(freq).fillna(0)
        test_df[f"{c}__freq"]  = test_df[c].fillna("missing").map(freq).fillna(0)

    return train_df, valid_df, test_df

In [8]:
# Multiclass target encoding (leakage-safe per fold)
# For multiclass, we create one encoded column per class.

def add_multiclass_target_encoding(train_df, valid_df, test_df, y_train, columns, classes, smoothing=20):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    global_priors = {}
    for cls in classes:
        global_priors[cls] = (y_train == cls).mean()

    for c in columns:
        tr_col = train_df[c].fillna("missing").astype(str)
        va_col = valid_df[c].fillna("missing").astype(str)
        te_col = test_df[c].fillna("missing").astype(str)

        stats = pd.DataFrame({"cat": tr_col, "y": y_train.values})

        for cls in classes:
            grp = stats.groupby("cat").apply(
                lambda g: ((g["y"] == cls).sum() + smoothing * global_priors[cls]) / (len(g) + smoothing)
            )
            feat_name = f"{c}__te_class_{cls}"

            train_df[feat_name] = tr_col.map(grp).fillna(global_priors[cls])
            valid_df[feat_name] = va_col.map(grp).fillna(global_priors[cls])
            test_df[feat_name]  = te_col.map(grp).fillna(global_priors[cls])

    return train_df, valid_df, test_df

In [9]:
# One-hot encoding for low-cardinality categorical columns

def add_onehot_low_cardinality(train_df, valid_df, test_df, columns, max_unique=8):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    low_card_cols = [c for c in columns if train_df[c].nunique(dropna=False) <= max_unique]

    if not low_card_cols:
        return train_df, valid_df, test_df, []

    train_cat = train_df[low_card_cols].fillna("missing").astype(str)
    valid_cat = valid_df[low_card_cols].fillna("missing").astype(str)
    test_cat  = test_df[low_card_cols].fillna("missing").astype(str)

    train_oh = pd.get_dummies(train_cat, prefix=low_card_cols)
    valid_oh = pd.get_dummies(valid_cat, prefix=low_card_cols)
    test_oh  = pd.get_dummies(test_cat, prefix=low_card_cols)

    all_cols = sorted(set(train_oh.columns) | set(valid_oh.columns) | set(test_oh.columns))

    train_oh = train_oh.reindex(columns=all_cols, fill_value=0)
    valid_oh = valid_oh.reindex(columns=all_cols, fill_value=0)
    test_oh  = test_oh.reindex(columns=all_cols, fill_value=0)

    train_df = pd.concat([train_df.drop(columns=low_card_cols), train_oh], axis=1)
    valid_df = pd.concat([valid_df.drop(columns=low_card_cols), valid_oh], axis=1)
    test_df  = pd.concat([test_df.drop(columns=low_card_cols), test_oh], axis=1)

    return train_df, valid_df, test_df, low_card_cols